In [53]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

In [55]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
from bs4 import BeautifulSoup


class DartsStatsScraper:
    def __init__(self):
        self.options = webdriver.ChromeOptions()
        self.options.add_argument('--disable-blink-features=AutomationControlled')
        self.options.add_argument('--headless')
        self.driver = webdriver.Chrome(options=self.options)

    def get_match_ids_and_stages(self, url):
        self.driver.get(url)

        WebDriverWait(self.driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "event__match"))
        )

        # Collect all stage and match elements
        stages = self.driver.find_elements(By.CLASS_NAME, "event__round--static")
        matches = self.driver.find_elements(By.CLASS_NAME, "event__match")

        match_ids_and_stages = []
        stage_index = 0  # Index to track the current stage

        for match in matches:
            # Check if the next stage should be used
            while stage_index < len(stages) - 1 and match.location['y'] > stages[stage_index + 1].location['y']:
                stage_index += 1  # Move to the next stage

            match_id = match.get_attribute("id")
            if match_id and match_id.startswith("g_"):
                match_ids_and_stages.append({
                    "match_id": match_id.split("_")[-1],
                    "stage": stages[stage_index].text.strip()
                })

        return match_ids_and_stages

    def get_match_stats(self, match_id, stage):
        def clean_stat_name(stat_name):
            """Cleans up the statistic name by making it lowercase and removing unwanted characters."""
            return stat_name.lower().replace(" ", "_").replace("(", "").replace(")", "")

        def extract_player_stats(soup, selector, default="Unknown"):
            """Extracts player names or other values based on the provided CSS selector."""
            container = soup.select_one(selector)
            return container.text.strip() if container else default

        stats_url = f"https://www.flashscore.com/match/{match_id}/#/match-summary/match-statistics/0"
        self.driver.get(stats_url)

        time.sleep(1)
        soup = BeautifulSoup(self.driver.page_source, 'html.parser')

        try:
            # Get Player 1 and Player 2 names
            player1 = extract_player_stats(soup, ".duelParticipant__home .participant__participantName.participant__overflow", "Unknown Player 1")
            player2 = extract_player_stats(soup, ".duelParticipant__away .participant__participantName.participant__overflow", "Unknown Player 2")

            # Get match score
            score_wrapper = soup.select_one(".detailScore__wrapper")
            if score_wrapper:
                score = score_wrapper.text.strip()
                sets = score.split("-")
                sets_player1 = int(sets[0].strip())
                sets_player2 = int(sets[1].strip())
            else:
                sets_player1 = sets_player2 = None

            # Get match date
            date = extract_player_stats(soup, ".duelParticipant__startTime", "Unknown Date")

            # Get match statistics
            stats = {}
            stat_rows = soup.select(".wcl-row_OFViZ")  
            for row in stat_rows:
                category = row.select_one(".wcl-category_7qsgP")
                if not category:
                    continue  
            
                stat_name = clean_stat_name(category.text.strip())
            
                # Extract stat values for each player
                value1 = row.select_one(".wcl-homeValue_-iJBW strong")
                value2 = row.select_one(".wcl-awayValue_rQvxs strong")
        
                value1_text = value1.text.strip() if value1 else None
                value2_text = value2.text.strip() if value2 else None
            
                # Only add stats if both values are found
                if value1_text and value2_text:
                    stats[f"{stat_name}_p1"] = value1_text
                    stats[f"{stat_name}_p2"] = value2_text

            # Create separate rows for each player
            def build_player_data(player, opponent, sets_won, sets_lost, suffix):
                return {
                    'match_date': date,
                    'stage': stage,
                    'player_name': player,
                    'opponent_name': opponent,
                    'sets_won': sets_won,
                    'sets_lost': sets_lost,
                    **{k.replace(suffix, ''): v for k, v in stats.items() if k.endswith(suffix)}
                }

            player1_data = build_player_data(player1, player2, sets_player1, sets_player2, '_p1')
            player2_data = build_player_data(player2, player1, sets_player2, sets_player1, '_p2')

            return [player1_data, player2_data]

        except Exception as e:
            print(f"Error scraping match {match_id}: {str(e)}")
            return None

    def scrape_all_matches(self, base_url):
        """Scrapes all the matches played in the tournament."""
        match_ids_and_stages = self.get_match_ids_and_stages(base_url)
        if not match_ids_and_stages:
            raise Exception("No match IDs found. The page structure might have changed.") # In case it changes in the future when you want to try it

        print(f"Found {len(match_ids_and_stages)} matches to scrape")
        all_stats = []

        for i, match_info in enumerate(match_ids_and_stages):
            match_id = match_info["match_id"]
            stage = match_info["stage"]
            print(f"Scraping match {i+1}/{len(match_ids_and_stages)} - Stage: {stage}")
            match_stats = self.get_match_stats(match_id, stage)
            if match_stats:
                all_stats.extend(match_stats)  # Add both rows (player1, player2)
            time.sleep(1)

        df = pd.DataFrame(all_stats)
        return df

    def close(self):
        self.driver.quit()


# Applying the scraper and saving it to a df
if __name__ == "__main__":
    url = "https://www.flashscore.com/darts/world/pdc-world-championship/results/"
    scraper = DartsStatsScraper()
    try:
        # Scrape all matches
        stats_df = scraper.scrape_all_matches(url)
        print(stats_df)
    finally:
        # Ensure the scraper is closed after use
        scraper.close()


C:\Users\henry\AppData\Local\Temp\ipykernel_27240\3490955055.py:36: DeprecationWarning: using WebElement.get_attribute() has been deprecated. Please use get_dom_attribute() instead.
  match_id = match.get_attribute("id")


Found 95 matches to scrape
Scraping match 1/95 - Stage: FINAL
Scraping match 2/95 - Stage: SEMI-FINALS
Scraping match 3/95 - Stage: SEMI-FINALS
Scraping match 4/95 - Stage: QUARTER-FINALS
Scraping match 5/95 - Stage: QUARTER-FINALS
Scraping match 6/95 - Stage: QUARTER-FINALS
Scraping match 7/95 - Stage: QUARTER-FINALS
Scraping match 8/95 - Stage: 1/8-FINALS
Scraping match 9/95 - Stage: 1/8-FINALS
Scraping match 10/95 - Stage: 1/8-FINALS
Scraping match 11/95 - Stage: 1/8-FINALS
Scraping match 12/95 - Stage: 1/8-FINALS
Scraping match 13/95 - Stage: 1/8-FINALS
Scraping match 14/95 - Stage: 1/8-FINALS
Scraping match 15/95 - Stage: 1/8-FINALS
Scraping match 16/95 - Stage: 1/16-FINALS
Scraping match 17/95 - Stage: 1/16-FINALS
Scraping match 18/95 - Stage: 1/16-FINALS
Scraping match 19/95 - Stage: 1/16-FINALS
Scraping match 20/95 - Stage: 1/16-FINALS
Scraping match 21/95 - Stage: 1/16-FINALS
Scraping match 22/95 - Stage: 1/16-FINALS
Scraping match 23/95 - Stage: 1/16-FINALS
Scraping match 24/

In [56]:
def process_checkouts(checkouts):

    # Extract percentage and counts
    percentage = float(checkouts.split('%')[0])  # Extract percentage as a float
    successful, total = map(int, checkouts.split('(')[1].strip(')').split('/'))  # Extract successful and total
    
    # Calculate failed checkouts
    failed = total - successful
    return percentage, successful, failed

# Apply the function to create new columns
stats_df[['checkout_percentage', 'successful_checkouts', 'failed_checkouts']] = stats_df['checkouts'].apply(
    lambda x: pd.Series(process_checkouts(x))
)

# Drop the original 'checkouts' column if no longer needed
stats_df.drop(columns=['checkouts'], inplace=True)

In [57]:
stats_df

,match_date,stage,player_name,opponent_name,sets_won,sets_lost,180_thrown,140+_thrown,100+_thrown,highest_checkout,average_3_darts,legs_won,checkout_percentage,successful_checkouts,failed_checkouts
0,03.01.2025 20:20,FINAL,Littler L.,van Gerwen M.,7,3,12,35,87,130,102.73,25,55.56,25.0,20.0
1,03.01.2025 20:20,FINAL,van Gerwen M.,Littler L.,3,7,13,35,79,132,100.69,14,36.84,14.0,24.0
2,02.01.2025 21:25,SEMI-FINALS,Bunting S.,Littler L.,1,6,9,34,66,80,100.10,12,36.36,12.0,21.0
3,02.01.2025 21:25,SEMI-FINALS,Littler L.,Bunting S.,6,1,13,37,77,170,105.48,19,44.19,19.0,24.0
4,02.01.2025 19:50,SEMI-FINALS,Dobey C.,van Gerwen M.,1,6,5,19,53,170,94.77,10,40.00,10.0,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,15.12.2024 21:20,1/64-FINALS,Barry K.,Huybrechts K.,3,1,7,15,41,112,94.97,11,50.00,11.0,11.0
184,15.12.2024 20:25,1/64-FINALS,Wattimena J.,Bellmont S.,3,0,2,10,27,95,98.54,9,45.00,9.0,11.0
185,15.12.2024 20:25,1/64-FINALS,Bellmont S.,Wattimena J.,0,3,5,11,25,83,92.95,4,36.36,4.0,7.0
186,15.12.2024 19:15,1/64-FINALS,Tricole T.,Comito J.,3,1,2,13,33,118,80.61,11,29.73,11.0,26.0


In [25]:
stats_df.to_excel('darts.xlsx', index=False)